In [2]:
# Inherits code from "multihazard_network_analysis_experiment_v1.ipynb"

# PURPOSE: integrate multi-hazard data and explore stresses created on the power grid

In [ ]:
import pandas as pd
import geopandas as gpd
import os, re
from shapely.geometry import Point, Polygon, MultiPolygon
import numpy as np

import numpy as np
from typing import Literal

from datetime import datetime, time
import geopandas as gpd
import xarray as xr


import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

import rioxarray

import requests
import json
from shapely.geometry import shape


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter

import networkx as nx



In [ ]:
# Construct power grid network

def get_start_end_coordinates(geom):
    start_points = []
    end_points = []
    for line in geom.geoms:  # Access individual LineString geometries within the MultiLineString
        start_points.append(line.coords[0])   # Get the first coordinate of each LineString
        end_points.append(line.coords[-1])    # Get the last coordinate of each LineString
    return start_points, end_points




In [ ]:
# Create tiles and map AOI to tiles

# =============================================================================
# Tile Generator: AOI → Tile Grid
# Supports square or hexagon grid generation over a GeoPandas AOI
# =============================================================================

def generate_tiles_over_aoi(
    aoi_gdf: gpd.GeoDataFrame,
    resolution_km: float = 50.0,
    shape: Literal["square", "hex"] = "square",
    crs: str = "EPSG:5070",
):
    """
    Generate a grid of square or hexagonal tiles fully covering an AOI.
    Only tiles that intersect the AOI are returned.

    Parameters
    ----------
    aoi_gdf : GeoDataFrame
        Input AOI polygons (any CRS). Can be multipolygon.
    resolution_km : float
        Target tile resolution in kilometers.
    shape : {"square", "hex"}
        Tile geometry type.
    crs : str
        Output / working CRS (must be projected). Default EPSG:5070.

    Returns
    -------
    tiles : GeoDataFrame
        Grid tiles intersecting the AOI.
    """
    import shapely.geometry as geom

    # --- Reproject AOI into equal‑area units
    aoi = aoi_gdf.to_crs(crs)
    minx, miny, maxx, maxy = aoi.total_bounds

    res_m = resolution_km * 1000.0

    # --- Buffer AOI extents so boundary tiles are not missed
    buffer = res_m * 2
    minx -= buffer
    miny -= buffer
    maxx += buffer
    maxy += buffer

    tiles = []

    if shape == "square":
        x_coords = np.arange(minx, maxx, res_m)
        y_coords = np.arange(miny, maxy, res_m)

        for x in x_coords:
            for y in y_coords:
                poly = geom.Polygon(
                    [
                        (x, y),
                        (x + res_m, y),
                        (x + res_m, y + res_m),
                        (x, y + res_m),
                    ]
                )
                tiles.append(poly)

    elif shape == "hex":
        w = res_m
        h = np.sqrt(3) * w / 2

        x_coords = np.arange(minx, maxx + w, w * 3/4)
        y_coords = np.arange(miny, maxy + h, h)

        for i, x in enumerate(x_coords):
            for j, y in enumerate(y_coords):
                # offset every other row
                x_offset = x + (w * 0.5 if j % 2 else 0)
                hex_poly = geom.Polygon(_hexagon_vertices(x_offset, y, w/2))
                tiles.append(hex_poly)

    else:
        raise ValueError("Shape must be 'square' or 'hex'")

    # --- Build GeoDataFrame and intersect with AOI
    grid = gpd.GeoDataFrame({"geometry": tiles}, crs=crs)
    grid = gpd.overlay(grid, aoi, how="intersection")

    # Assign tile IDs
    grid["tile_id"] = [f"T{i:07d}" for i in range(len(grid))]

    return grid


def _hexagon_vertices(x_center: float, y_center: float, r: float):
    """Return coordinates for a hexagon centered at (x, y) with radius r."""
    angles = np.linspace(0, 2 * np.pi, num=7)[:-1]
    return [(x_center + r * np.cos(a), y_center + r * np.sin(a)) for a in angles]


# Get power grid as our background network

In [ ]:
file_path_powergrid = "/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/physical_grid_data/U.S._Electric_Power_Transmission_Lines.geojson"
gdf_powergrid = gpd.read_file(file_path_powergrid)



# Convert timeframes to folium-friendly types
gdf_powergrid['SOURCEDATE'] = pd.to_datetime(gdf_powergrid['SOURCEDATE']).dt.strftime('%Y-%m-%dT%H:%M:%S')
gdf_powergrid['VAL_DATE'] = pd.to_datetime(gdf_powergrid['VAL_DATE']).dt.strftime('%Y-%m-%dT%H:%M:%S')


# Drop columns with 'NOT AVAILABLE' as substation value
# gdf[ (gdf['SUB_1']=='NOT AVAILABLE') | (gdf['SUB_2']=='NOT AVAILABLE')]
condition = ( (gdf_powergrid['SUB_1']=='NOT AVAILABLE') | (gdf_powergrid['SUB_2']=='NOT AVAILABLE') )
gdf_powergrid = gdf_powergrid[~condition]
# gdf

# Drop columns with 'NONE' as substation value
condition = ( (gdf_powergrid['SUB_1']=='NONE') | (gdf_powergrid['SUB_2']=='NONE') )
gdf_powergrid = gdf_powergrid[~condition]
gdf_powergrid



# Define area of interest


In [ ]:
file_path_aoi = "/Users/ryanmc/Documents/Complex_Risk_Science/dev/Complex-Risk-Collective/.github/Projects/NASA-disasters-grid-resilience/data/selected_states_MarchApril2023_event.geojson"
gdf_aoi = gpd.read_file(file_path_aoi)
gdf_aoi = gdf_aoi.to_crs(gdf_powergrid.crs)

In [ ]:
# Clip to ensure only grid elements in the AOI are kept
gdf_powergrid_aoi = gpd.sjoin(gdf_powergrid, gdf_aoi, how="inner", predicate='within').drop(columns=['index_right'])

In [ ]:
# First, check the CRS and bounds of the power grid AOI
print("Power grid AOI info:")
print(f"  CRS: {gdf_powergrid_aoi.crs}")
print(f"  Bounds: {gdf_powergrid_aoi.total_bounds}")
print(f"  Number of features: {len(gdf_powergrid_aoi)}")

# Generate tiles over the power grid AOI
# Note: Using the dissolved union of the AOI for better tile coverage
tiles_gdf = generate_tiles_over_aoi(
    aoi_gdf=gdf_aoi,  # Use the original AOI polygon instead of the power grid lines
    resolution_km=50.0,  # 50 km tiles
    shape="square",      # or "hex" for hexagonal tiles
    crs="EPSG:5070"      # Equal-area projection for US
)

print(f"\nGenerated {len(tiles_gdf)} tiles covering the AOI")
if len(tiles_gdf) > 0:
    print(f"Tile CRS: {tiles_gdf.crs}")
    print(f"Tile bounds: {tiles_gdf.total_bounds}")
    tiles_gdf.head()
else:
    print("WARNING: No tiles generated!")

In [ ]:
target_crs = 'EPSG:4326'  # Keep in lat/lon for consistency with source data

print(f"\n{'='*60}")
print(f"Standardizing all geodataframes to {target_crs}")
print(f"{'='*60}")

# Reproject tiles_gdf to match
if tiles_gdf.crs != target_crs:
    print(f"Reprojecting tiles_gdf from {tiles_gdf.crs} to {target_crs}...")
    tiles_gdf = tiles_gdf.to_crs(target_crs)
    print("✓ tiles_gdf reprojected")

# Ensure gdf_aoi matches (should already be 4326)
if gdf_aoi.crs != target_crs:
    print(f"Reprojecting gdf_aoi from {gdf_aoi.crs} to {target_crs}...")
    gdf_aoi = gdf_aoi.to_crs(target_crs)
    print("✓ gdf_aoi reprojected")

print(f"\nFinal CRS verification:")
print(f"  gdf_aoi: {gdf_aoi.crs}")
print(f"  tiles_gdf: {tiles_gdf.crs}")
print(f"  gdf_powergrid_aoi: {gdf_powergrid_aoi.crs}")

In [ ]:
#Plotting
powergrid_aoi_map = gdf_powergrid_aoi.plot(figsize=(20, 16),linewidth=0.5)


# Calculate our power grid network 

In [ ]:
# Apply the start and end points function to the MultiLineString geometry
gdf_powergrid_aoi['start_points'], gdf_powergrid_aoi['end_points'] = zip(*gdf_powergrid_aoi['geometry'].apply(lambda geom: get_start_end_coordinates(geom)))


In [ ]:
from shapely.geometry import Point, LineString

# Create a NetworkX graph
G = nx.Graph()

# Add edges to the graph
for idx, row in gdf_powergrid_aoi.iterrows():
    start = tuple(row['start_points'])
    end = tuple(row['end_points'])
    start_id = f"{start[0][0]}_{start[0][1]}"
    end_id = f"{end[0][0]}_{end[0][1]}"
    G.add_edge(start_id, end_id, geometry=LineString([start[0], end[0]]))


# Calculate degree centrality
centrality = nx.degree_centrality(G)

# Get the top X nodes based on degree centrality
num_nodes = 1000
top_X_nodes = sorted(centrality, key=centrality.get, reverse=True)[:num_nodes]

# Get the subgraph of the top X nodes
subgraph = G.subgraph(top_X_nodes)

# Create a GeoDataFrame of the edges for plotting
edges = []
for u, v, data in subgraph.edges(data=True):
    start_coords = tuple(map(float, u.split('_')))
    end_coords = tuple(map(float, v.split('_')))
    edges.append(LineString([start_coords, end_coords]))

edges_gdf = gpd.GeoDataFrame(geometry=edges)

# Create a GeoDataFrame of the nodes for plotting
nodes = [Point(tuple(map(float, node.split('_')))) for node in top_X_nodes]
nodes_gdf = gpd.GeoDataFrame(geometry=nodes)



# Read in and format the Space Weather Data
Two options: 
- SuperMAG 
- Raw geoE data from IRIS shared by Greg Lucas 

both are location-specific (not gridded) 

In [ ]:
file_path_geoE = '/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/Lucas_GeoElectricFields_March-April2023/mcgranaghan-rawdata.nc'
os.path.exists(file_path_geoE)

In [ ]:
# Load, focus on Eh, subset to Mar 31–Apr 4, and reproject to EPSG of the generated tiles if needed

# 1) Open dataset
fp_geoE = os.path.expanduser(file_path_geoE) if isinstance(file_path_geoE, str) else str(file_path_geoE)
ds_geoE = xr.open_dataset(fp_geoE, decode_times=True)

# 2) Identify coordinate and time names
lat_name = next((n for n in ['lat','latitude','Latitude','y','Y'] if n in ds_geoE.coords or n in ds_geoE.dims), None)
lon_name = next((n for n in ['lon','longitude','Longitude','x','X'] if n in ds_geoE.coords or n in ds_geoE.dims), None)
time_name = next((n for n in ['time','Time','valid_time','datetime'] if (n in ds_geoE.coords) or (n in ds_geoE.dims)), None)

# 3) Get or build Eh
if 'Eh' in ds_geoE.data_vars:
    Eh = ds_geoE['Eh']
else:
    # Try to compute |E| = sqrt(Ex^2 + Ey^2) from available names
    ex_candidates = ['Ex','E_x','east','E_east','E_e','E_lon','E_longitude']
    ey_candidates = ['Ey','E_y','north','E_north','E_n','E_lat','E_latitude']
    ex_name = next((n for n in ex_candidates if n in ds_geoE.data_vars), None)
    ey_name = next((n for n in ey_candidates if n in ds_geoE.data_vars), None)
    if ex_name and ey_name:
        Eh = np.hypot(ds_geoE[ex_name], ds_geoE[ey_name])
        Eh.name = 'Eh'
        Eh.attrs.update({'long_name':'Horizontal electric field magnitude','units':ds_geoE[ex_name].attrs.get('units','')})
    else:
        # Fallback: pick the first numeric var and treat it as Eh
        numeric_vars = [k for k,v in ds_geoE.data_vars.items() if np.issubdtype(v.dtype, np.number)]
        if numeric_vars:
            Eh = ds_geoE[numeric_vars[0]]
            Eh.name = 'Eh'
        else:
            raise ValueError("Could not find 'Eh' or Ex/Ey to compute it.")

# 4) Subset to Mar 31–Apr 4 across whatever year is present
if time_name and (time_name in Eh.dims):
    tcoord = Eh[time_name]
    # mask days Mar 31 through Apr 4 (inclusive), regardless of year
    mask = ((tcoord.dt.month==3) & (tcoord.dt.day>=31)) | ((tcoord.dt.month==4) & (tcoord.dt.day<=4))
    if mask.any():
        Eh_sub = Eh.sel({time_name: tcoord[mask]})
    else:
        print("Warning: No timestamps found for Mar 31–Apr 4; using full time range.")
        Eh_sub = Eh
else:
    Eh_sub = Eh

# 5) Reproject to EPSG of the generated tiles when applicable
Eh_4326 = Eh_sub
is_grid = (lat_name in Eh_sub.dims) and (lon_name in Eh_sub.dims)
if is_grid:
    try:
        import rioxarray  # noqa: F401 adds .rio accessor
        # set spatial dims and assign a CRS if missing
        Eh_spatial = Eh_sub.rio.set_spatial_dims(x_dim=lon_name, y_dim=lat_name, inplace=False)
        # Heuristic: if lon/lat look like degrees, set EPSG:4326; else try grid_mapping metadata
        try:
            lon_vals = np.asarray(ds_geoE[lon_name].values)
            lat_vals = np.asarray(ds_geoE[lat_name].values)
            looks_like_degrees = (np.nanmax(np.abs(lon_vals)) <= 360) and (np.nanmax(np.abs(lat_vals)) <= 90)
        except Exception:
            looks_like_degrees = True
        if looks_like_degrees:
            Eh_spatial = Eh_spatial.rio.write_crs("EPSG:4326", inplace=False)
        else:
            gm = Eh_spatial.attrs.get('grid_mapping')
            if gm and (gm in ds_geoE):
                crs_wkt = ds_geoE[gm].attrs.get('spatial_ref') or ds_geoE[gm].attrs.get('crs_wkt')
                if crs_wkt:
                    Eh_spatial = Eh_spatial.rio.write_crs(crs_wkt, inplace=False)
        # If already 4326, this is a no-op; otherwise reprojects
        Eh_4326 = Eh_spatial.rio.reproject("EPSG:4326", nodata=np.nan)
    except Exception as e:
        print("Reprojection skipped or failed:", e)
        Eh_4326 = Eh_sub

print("Eh_4326 dims:", Eh_4326.dims)
print("Eh_4326 coords:", list(Eh_4326.coords))

In [ ]:
# Map geoelectric field data to tiles for H(time, tile_id)



# 2) Extract station/site locations from the geoE dataset
# Detect station dimension
station_dim = next((d for d in ['site','station','stations','id','station_id','point','points'] 
                     if d in ds_geoE.dims), None)

if station_dim is None and 'site' in ds_geoE.coords:
    station_dim = ds_geoE['site'].dims[0]

if station_dim is None:
    raise ValueError("Could not identify station dimension in geoE dataset")

# Get lat/lon coordinates for each site
lat_coord = next((n for n in ['lat','latitude','Latitude','station_lat'] if n in ds_geoE.coords), None)
lon_coord = next((n for n in ['lon','longitude','Longitude','station_lon'] if n in ds_geoE.coords), None)

if lat_coord is None or lon_coord is None:
    raise ValueError(f"Could not find lat/lon coordinates. Available coords: {list(ds_geoE.coords)}")

# Create GeoDataFrame of station locations
site_lats = ds_geoE[lat_coord].values
site_lons = ds_geoE[lon_coord].values

# Get site IDs if available
if 'site' in ds_geoE.coords:
    site_ids = ds_geoE['site'].values
elif station_dim in ds_geoE.coords:
    site_ids = ds_geoE[station_dim].values
else:
    site_ids = np.arange(len(site_lats))

sites_gdf = gpd.GeoDataFrame({
    'site_id': site_ids,
    'geometry': [Point(lon, lat) for lon, lat in zip(site_lons, site_lats)]
}, crs="EPSG:4326")

print(f"Found {len(sites_gdf)} geoE measurement sites")

# 3) Spatial join to assign each site to a tile
sites_with_tiles = gpd.sjoin(sites_gdf, tiles_4326[['tile_id', 'geometry']], 
                               how='left', predicate='within')

print(f"Assigned {sites_with_tiles['tile_id'].notna().sum()} sites to tiles")
print(f"Sites outside tiles: {sites_with_tiles['tile_id'].isna().sum()}")

# 4) Build mapping from site to tile_id
site_to_tile = dict(zip(sites_with_tiles['site_id'], sites_with_tiles['tile_id']))

# 5) Create tile-aggregated time series: max Eh per tile per time
time_dim = next((n for n in ['time','Time','valid_time','datetime'] if n in Eh_4326.dims), None)

if time_dim is None:
    raise ValueError("No time dimension found in Eh_4326")

# Extract time values
time_values = pd.to_datetime(Eh_4326[time_dim].values)

# Initialize dictionary to collect data: {tile_id: {time: max_Eh}}
tile_time_data = {}

print("Aggregating Eh data by tile and time...")
# Iterate over each site and aggregate to tiles
for i, site_id in enumerate(site_ids):
    tile_id = site_to_tile.get(site_id)
    if pd.isna(tile_id):
        continue  # Site not in any tile
    
    # Extract Eh time series for this site
    try:
        site_eh = Eh_4326.isel({station_dim: i}).values
    except Exception:
        continue
    
    # Initialize tile entry if needed
    if tile_id not in tile_time_data:
        tile_time_data[tile_id] = {}
    
    # For each time step, take max across sites in the tile
    for t_idx, (t, eh_val) in enumerate(zip(time_values, site_eh)):
        if np.isnan(eh_val):
            continue
        if t not in tile_time_data[tile_id]:
            tile_time_data[tile_id][t] = eh_val
        else:
            tile_time_data[tile_id][t] = max(tile_time_data[tile_id][t], eh_val)

# 6) Convert to DataFrame: H(time, tile_id)
records = []
for tile_id, time_dict in tile_time_data.items():
    for t, eh_max in time_dict.items():
        records.append({'time': t, 'tile_id': tile_id, 'Eh_max': eh_max})

H_geoE = pd.DataFrame(records)

# Pivot to get H(time, tile_id) format
H_geoE_pivot = H_geoE.pivot(index='time', columns='tile_id', values='Eh_max')

print(f"\nCreated hazard state vector H_geoE:")
print(f"  Shape: {H_geoE_pivot.shape} (time steps × tiles)")
print(f"  Time range: {H_geoE_pivot.index.min()} to {H_geoE_pivot.index.max()}")
print(f"  Tiles with data: {H_geoE_pivot.notna().any().sum()} / {len(tiles_gdf)}")
print(f"  Coverage: {100 * H_geoE_pivot.notna().sum().sum() / H_geoE_pivot.size:.1f}% non-null")

display(H_geoE_pivot.head())


# Read in and format the Terrestrial Weather Data (for now SWDI gridded)

In [ ]:
file_path_terrestrial_weather = os.path.expanduser('~/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/swdi_nx3_hourly_50km_conus.nc')
os.path.exists(file_path_terrestrial_weather)

In [ ]:
# Load and explore SWDI terrestrial weather data

fp_swdi = os.path.expanduser(file_path_terrestrial_weather)
print("Using SWDI NetCDF path:", fp_swdi)

if not os.path.exists(fp_swdi):
    raise FileNotFoundError(f"SWDI NetCDF file not found: {fp_swdi}")

# Open dataset
ds_swdi = xr.open_dataset(fp_swdi, decode_times=True)
display(ds_swdi)

print("\nData variables:", list(ds_swdi.data_vars))
print("Coordinates:", list(ds_swdi.coords))
print("Dimensions:", {k: int(v) for k, v in ds_swdi.dims.items()})

# Check if nx3structure_counts exists
if 'nx3structure_counts' in ds_swdi.data_vars:
    print("\n✓ Found 'nx3structure_counts' variable")
    print(f"  Shape: {ds_swdi['nx3structure_counts'].shape}")
    print(f"  Dims: {ds_swdi['nx3structure_counts'].dims}")
else:
    print(f"\n⚠ 'nx3structure_counts' not found. Available variables: {list(ds_swdi.data_vars)}")


In [ ]:
# Start from dataset with nanosecond-decoded time
ds_src = ds_swdi_ns if 'ds_swdi_ns' in globals() else ds_swdi
var = ds_src['nx3structure_counts']

# Identify spatial dims: expect y/x on EPSG:5070
dims = list(var.dims)
print(f"SWDI dims: {dims}")

# Prefer canonical y,x names if present; otherwise infer by excluding time dim
time_dim_name = swdi_time if 'swdi_time' in globals() else next(d for d in dims if 'time' in d.lower())
y_dim = 'y' if 'y' in dims else next(d for d in dims if d != time_dim_name)
x_dim = 'x' if 'x' in dims else next(d for d in dims if d not in (time_dim_name, y_dim))
print(f"Using spatial dims y='{y_dim}', x='{x_dim}'")

# SWDI: Explicitly set EPSG:5070 (Albers) for x/y and reproject to WGS84 (EPSG:4326)

# Attach CRS EPSG:5070 and set spatial dims
var_spatial = var.rio.set_spatial_dims(x_dim=x_dim, y_dim=y_dim, inplace=False)
var_spatial = var_spatial.rio.write_crs("EPSG:5070", inplace=False)

# Ensure floating dtype and nodata is representable
if not np.issubdtype(var_spatial.dtype, np.floating):
    print(f"Casting SWDI data from {var_spatial.dtype} to float32 for reprojection…")
    var_spatial = var_spatial.astype('float32')

# Write nodata before reprojection
var_spatial = var_spatial.rio.write_nodata(np.nan, inplace=False)

# Reproject to WGS84 with safe nodata handling
print("Reprojecting SWDI grid from EPSG:5070 to EPSG:4326…")
try:
    var_wgs84 = var_spatial.rio.reproject("EPSG:4326", nodata=np.nan, resampling=0)
except ValueError as e:
    print(f"Reproject with NaN nodata failed ({e}). Retrying with sentinel nodata…")
    var_spatial = var_spatial.rio.write_nodata(-9999.0, inplace=False)
    var_wgs84 = var_spatial.rio.reproject("EPSG:4326", nodata=-9999.0, resampling=0)
    var_wgs84 = var_wgs84.where(var_wgs84 != -9999.0, np.nan)

# Extract lon/lat coordinate vectors
lon_vals = var_wgs84[x_dim].values
lat_vals = var_wgs84[y_dim].values
print(f"lon range: {np.nanmin(lon_vals):.3f} .. {np.nanmax(lon_vals):.3f}")
print(f"lat range: {np.nanmin(lat_vals):.3f} .. {np.nanmax(lat_vals):.3f}")

# Subset time window Mar 31–Apr 4
swdi_time_name = time_dim_name
time_vals = pd.to_datetime(var_wgs84[swdi_time_name].values)
mask = ((time_vals.month==3) & (time_vals.day>=31)) | ((time_vals.month==4) & (time_vals.day<=4))
if mask.any():
    var_wgs84_sub = var_wgs84.sel({swdi_time_name: var_wgs84[swdi_time_name].values[mask]})
    print(f"Subset timesteps: {mask.sum()} (Mar31–Apr4)")
else:
    var_wgs84_sub = var_wgs84
    print("No Mar31–Apr4 timesteps; using full range.")


# Build spatial index for tiles
sidx = tiles_gdf.sindex

# Map grid centers to tiles
print("Mapping grid centers to tiles…")
grid_to_tile = {}
for i, lat in enumerate(lat_vals):
    for j, lon in enumerate(lon_vals):
        pt = Point(float(lon), float(lat))
        cand = list(sidx.intersection(pt.bounds))
        for k in cand:
            if tiles_gdf.iloc[k].geometry.contains(pt):
                grid_to_tile[(i, j)] = tiles_gdf.iloc[k]['tile_id']
                break
print(f"Mapped {len(grid_to_tile)} grid centers to tiles")

# Aggregate counts by tile and time
print("Aggregating SWDI to tiles…")
agg = {}
for t_idx in range(var_wgs84_sub.sizes[swdi_time_name]):
    tval = pd.to_datetime(var_wgs84_sub[swdi_time_name].values[t_idx])
    arr = var_wgs84_sub.isel({swdi_time_name: t_idx}).values
    # Expect arr shape (y, x)
    for (i, j), tile_id in grid_to_tile.items():
        val = arr[i, j] if arr.ndim == 2 else np.nan
        if not np.isnan(val):
            agg.setdefault(tile_id, {}).setdefault(tval, 0.0)
            agg[tile_id][tval] += float(val)

# Build pivot dataframe
rows = []
for tile_id, td in agg.items():
    for t, s in td.items():
        rows.append({'time': t, 'tile_id': tile_id, 'nx3_counts': s})
H_SWDI_reproj = pd.DataFrame(rows)

if not H_SWDI_reproj.empty:
    H_SWDI_pivot = H_SWDI_reproj.pivot(index='time', columns='tile_id', values='nx3_counts').sort_index()
    print("\nSWDI pivot (EPSG:5070→4326) summary:")
    print(f"  Shape: {H_SWDI_pivot.shape}")
    print(f"  Time range: {H_SWDI_pivot.index.min()} to {H_SWDI_pivot.index.max()}")
    print(f"  Tiles with data: {H_SWDI_pivot.notna().any().sum()} / {len(tiles_gdf)}")
else:
    print("⚠ Empty SWDI aggregation after reprojection—check dims and ranges.")

# Read in and format the Wildfire Data (for now MODIS point data)


In [ ]:
# Load and explore MODIS wildfire data

file_path_wildfire = '/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/wildfire_data/DL_FIRE_M-C61_579069 - Linzheng data UW Capstone 2024-2025/fire_archive_M-C61_579069.shp'

print("Using wildfire shapefile path:", file_path_wildfire)

if not os.path.exists(file_path_wildfire):
    raise FileNotFoundError(f"Wildfire shapefile not found: {file_path_wildfire}")

# Read shapefile with GeoPandas
gdf_fires = gpd.read_file(file_path_wildfire)

print(f"\nLoaded {len(gdf_fires)} fire detections")
print(f"CRS: {gdf_fires.crs}")
print(f"\nColumns: {list(gdf_fires.columns)}")

# Check for required columns
if 'FRP' in gdf_fires.columns:
    print("\n✓ Found 'FRP' (Fire Radiative Power) column")
    print(f"  FRP range: {gdf_fires['FRP'].min():.2f} to {gdf_fires['FRP'].max():.2f} MW")
else:
    print(f"\n⚠ 'FRP' not found. Available columns: {list(gdf_fires.columns)}")

# Check for time/date column
time_cols = [c for c in gdf_fires.columns if any(x in c.lower() for x in ['time', 'date', 'acq'])]
print(f"\nPotential time columns: {time_cols}")

display(gdf_fires.head())

#  some important items from the readme.txt at https://firms.modaps.eosdis.nasa.gov/download/Readme.txt

# -- fire_archive_M-C61_xx = MODIS standard quality Thermal Anomalies / Fire locations 
#                         processed by the University of Maryland with a 3-month
#                         lag and distributed by FIRMS. These standard data (MCD14ML) 
#                         replace the NRT (MCD14DL) files when available.

# The MODIS and VIIRS fire/hotspot data supplied to you are in the WGS84 Geographic
# projection (the "latitude/longitude projection").

# Attributes of the data described at https://www.earthdata.nasa.gov/data/tools/firms/active-fire-data-attributes-modis-viirs

# Will use Fire Radiative Power (FRP) for now


In [ ]:
# Map wildfire data to tiles for H(time, tile_id)

# 1) Ensure CRS matches tiles (should be WGS84/EPSG:4326)
if gdf_fires.crs != 'EPSG:4326':
    gdf_fires_4326 = gdf_fires.to_crs('EPSG:4326')
else:
    gdf_fires_4326 = gdf_fires

# 2) Identify and parse time column
# Common MODIS column names: 'ACQ_DATE', 'ACQ_TIME', or combined datetime
time_col = next((c for c in gdf_fires_4326.columns if 'ACQ_DATE' in c.upper()), None)
time_time_col = next((c for c in gdf_fires_4326.columns if 'ACQ_TIME' in c.upper()), None)

if time_col:
    print(f"Found time column: {time_col}")
    if time_time_col:
        print(f"Found time-of-day column: {time_time_col}")
        # Combine date and time
        gdf_fires_4326['datetime'] = pd.to_datetime(
            gdf_fires_4326[time_col].astype(str) + ' ' + gdf_fires_4326[time_time_col].astype(str).str.zfill(4),
            format='%Y-%m-%d %H%M',
            errors='coerce'
        )
    else:
        gdf_fires_4326['datetime'] = pd.to_datetime(gdf_fires_4326[time_col], errors='coerce')
else:
    raise ValueError(f"Could not find time column. Available columns: {list(gdf_fires_4326.columns)}")

# Drop rows with invalid datetimes
gdf_fires_4326 = gdf_fires_4326[gdf_fires_4326['datetime'].notna()].copy()

print(f"\nTime range: {gdf_fires_4326['datetime'].min()} to {gdf_fires_4326['datetime'].max()}")

# 3) Subset to Mar 31 - Apr 4 timeframe
mask = ((gdf_fires_4326['datetime'].dt.month==3) & (gdf_fires_4326['datetime'].dt.day>=31)) | \
       ((gdf_fires_4326['datetime'].dt.month==4) & (gdf_fires_4326['datetime'].dt.day<=4))

gdf_fires_subset = gdf_fires_4326[mask].copy()
print(f"Subset to {len(gdf_fires_subset)} fire detections (Mar 31 - Apr 4)")



fires_with_tiles = gpd.sjoin(gdf_fires_subset, tiles_gdf[['tile_id', 'geometry']], 
                               how='left', predicate='within')

print(f"Assigned {fires_with_tiles['tile_id'].notna().sum()} fires to tiles")
print(f"Fires outside tiles: {fires_with_tiles['tile_id'].isna().sum()}")

# 5) Aggregate FRP by tile and time (hourly bins)
# Round to nearest hour for aggregation
fires_with_tiles['time_hour'] = fires_with_tiles['datetime'].dt.floor('H')

# Group by tile and time, sum FRP
H_wildfire_raw = fires_with_tiles.groupby(['time_hour', 'tile_id'])['FRP'].sum().reset_index()
H_wildfire_raw.columns = ['time', 'tile_id', 'FRP_sum']

# Pivot to H(time, tile_id) format
H_wildfire_pivot = H_wildfire_raw.pivot(index='time', columns='tile_id', values='FRP_sum')

print(f"\nCreated hazard state vector H_wildfire:")
print(f"  Shape: {H_wildfire_pivot.shape} (time steps × tiles)")
print(f"  Time range: {H_wildfire_pivot.index.min()} to {H_wildfire_pivot.index.max()}")
print(f"  Tiles with data: {H_wildfire_pivot.notna().any().sum()} / {len(tiles_gdf)}")
print(f"  Coverage: {100 * H_wildfire_pivot.notna().sum().sum() / H_wildfire_pivot.size:.1f}% non-null")
print(f"  Total FRP: {H_wildfire_pivot.sum().sum():.2f} MW")

display(H_wildfire_pivot.head())


# Combine all hazard components into a unified multihazard state vector


In [ ]:

print("=" * 80)
print("MULTIHAZARD STATE VECTOR SUMMARY")
print("=" * 80)

# Check what we have
hazard_components = {}

if 'H_geoE_pivot' in globals():
    hazard_components['Space Weather (Eh)'] = H_geoE_pivot
    print(f"\n✓ Space Weather (geoelectric field):")
    print(f"    Shape: {H_geoE_pivot.shape}")
    print(f"    Time: {H_geoE_pivot.index.min()} to {H_geoE_pivot.index.max()}")
    print(f"    Coverage: {100 * H_geoE_pivot.notna().sum().sum() / H_geoE_pivot.size:.1f}%")

if 'H_SWDI_pivot' in globals():
    hazard_components['Terrestrial Weather (NX3)'] = H_SWDI_pivot
    print(f"\n✓ Terrestrial Weather (SWDI nx3):")
    print(f"    Shape: {H_SWDI_pivot.shape}")
    print(f"    Time: {H_SWDI_pivot.index.min()} to {H_SWDI_pivot.index.max()}")
    print(f"    Coverage: {100 * H_SWDI_pivot.notna().sum().sum() / H_SWDI_pivot.size:.1f}%")

if 'H_wildfire_pivot' in globals():
    hazard_components['Wildfire (FRP)'] = H_wildfire_pivot
    print(f"\n✓ Wildfire (MODIS FRP):")
    print(f"    Shape: {H_wildfire_pivot.shape}")
    print(f"    Time: {H_wildfire_pivot.index.min()} to {H_wildfire_pivot.index.max()}")
    print(f"    Coverage: {100 * H_wildfire_pivot.notna().sum().sum() / H_wildfire_pivot.size:.1f}%")

print(f"\n" + "=" * 80)
print(f"Total hazard components: {len(hazard_components)}")

# Create a unified time index (union of all time indices)
if hazard_components:
    all_times = set()
    for df in hazard_components.values():
        all_times.update(df.index)
    
    unified_time_index = pd.DatetimeIndex(sorted(all_times))
    print(f"\nUnified time index:")
    print(f"  Total timesteps: {len(unified_time_index)}")
    print(f"  Range: {unified_time_index.min()} to {unified_time_index.max()}")
    
    # Reindex all components to unified time index
    H_components_aligned = {}
    for name, df in hazard_components.items():
        H_components_aligned[name] = df.reindex(unified_time_index)
    
    print(f"\n✓ All hazard components aligned to unified time index")
    
    # Optional: Create a combined multi-index DataFrame
    # This stacks all hazards into one structure: H(time, tile_id, hazard_type)
    H_combined = pd.concat(H_components_aligned, axis=1, keys=hazard_components.keys())
    
    print(f"\nCombined multihazard state vector:")
    print(f"  Shape: {H_combined.shape}")
    print(f"  Columns: {H_combined.columns.nlevels} levels (hazard type, tile_id)")
    print(f"  Index: {len(H_combined.index)} timesteps")
else:
    print("\n⚠ No hazard components found. Run the data loading cells first.")

print("=" * 80)


# Power Grid Network Validation & Hazard Mapping

Validate network representation and map hazard tiles to transmission components

In [ ]:
# Network topology diagnostics
print("=== Power Grid Network Summary ===")
print(f"Total nodes (substations): {G.number_of_nodes()}")
print(f"Total edges (transmission lines): {G.number_of_edges()}")
print(f"Connected: {nx.is_connected(G)}")
if not nx.is_connected(G):
    components = list(nx.connected_components(G))
    print(f"Number of connected components: {len(components)}")
    print(f"Largest component size: {max(len(c) for c in components)} nodes")
    print(f"Smallest component size: {min(len(c) for c in components)} nodes")

# Degree distribution
degrees = [G.degree(n) for n in G.nodes()]
print(f"\nNode degree statistics:")
print(f"  Min degree: {min(degrees)}")
print(f"  Max degree: {max(degrees)}")
print(f"  Mean degree: {np.mean(degrees):.2f}")
print(f"  Median degree: {np.median(degrees):.1f}")

# High-degree nodes (likely major substations)
high_degree = [(n, G.degree(n)) for n in G.nodes() if G.degree(n) >= 4]
print(f"\nHigh-connectivity substations (degree ≥4): {len(high_degree)}")
if high_degree:
    print(f"  Sample (top 5 by degree):")
    for node, deg in sorted(high_degree, key=lambda x: x[1], reverse=True)[:5]:
        print(f"    Node {node}: {deg} connections")

In [ ]:
# Build complete node and edge GeoDataFrames (WGS84) for the entire network
# Nodes: parse node IDs (lon_lat) to Point geometries
node_geoms = []
node_ids = []
for node_id in G.nodes():
    try:
        lon_str, lat_str = node_id.split('_')
        lon, lat = float(lon_str), float(lat_str)
        node_geoms.append(Point(lon, lat))
        node_ids.append(node_id)
    except Exception as e:
        print(f"Warning: could not parse node {node_id}: {e}")
        
nodes_gdf = gpd.GeoDataFrame({
    'node_id': node_ids,
    'degree': [G.degree(n) for n in node_ids]
}, geometry=node_geoms, crs='EPSG:4326')

# Edges: reconstruct LineString from node IDs
edge_geoms = []
edge_ids = []
edge_start = []
edge_end = []
for u, v in G.edges():
    try:
        u_lon, u_lat = map(float, u.split('_'))
        v_lon, v_lat = map(float, v.split('_'))
        edge_geoms.append(LineString([(u_lon, u_lat), (v_lon, v_lat)]))
        edge_ids.append(f"{u}|{v}")
        edge_start.append(u)
        edge_end.append(v)
    except Exception as e:
        print(f"Warning: could not parse edge ({u}, {v}): {e}")

edges_gdf_full = gpd.GeoDataFrame({
    'edge_id': edge_ids,
    'start_node': edge_start,
    'end_node': edge_end,
}, geometry=edge_geoms, crs='EPSG:4326')

print(f"Built GeoDataFrames: {len(nodes_gdf)} nodes, {len(edges_gdf_full)} edges")

In [ ]:
# Spatial join: Map nodes (substations) to hazard tiles
# Each node gets assigned to the tile containing it (or nearest if outside)
# nodes_with_tiles = gpd.sjoin(nodes_gdf, tiles_gdf[['tile_id', 'geometry']], how='left', predicate='within')
nodes_with_tiles = gpd.sjoin(nodes_gdf, tiles_gdf[['tile_id', 'geometry']], how='left', predicate='within')

# For nodes outside tiles (edge cases), use nearest
nodes_outside = nodes_with_tiles[nodes_with_tiles['tile_id'].isna()]
if len(nodes_outside) > 0:
    print(f"Found {len(nodes_outside)} nodes outside tiles; assigning to nearest tile...")
    for idx in nodes_outside.index:
        node_geom = nodes_gdf.loc[idx, 'geometry']
        distances = tiles_gdf.geometry.distance(node_geom)
        nearest_tile = tiles_gdf.loc[distances.idxmin(), 'tile_id']
        nodes_with_tiles.loc[idx, 'tile_id'] = nearest_tile

# Build node-to-tile mapping
node_to_tile = nodes_with_tiles[['node_id', 'tile_id']].set_index('node_id')['tile_id'].to_dict()
print(f"Mapped {len(node_to_tile)} nodes to {len(set(node_to_tile.values()))} unique tiles")

# Preview
print("\nSample node→tile mappings:")
for node_id, tile_id in list(node_to_tile.items())[:5]:
    print(f"  Node {node_id} → Tile {tile_id}")

In [ ]:
# Spatial join: Map edges (transmission lines) to hazard tiles
# Strategy: each line may intersect multiple tiles; assign all intersecting tiles
# Use spatial index for efficiency
edge_to_tiles = {}
# tile_sindex = tiles_gdf.sindex
tile_sindex = tiles_gdf.sindex

for idx, row in edges_gdf_full.iterrows():
    edge_id = row['edge_id']
    edge_geom = row['geometry']
    # Find candidate tiles via bbox
    possible_matches_idx = list(tile_sindex.intersection(edge_geom.bounds))
    # Refine with actual intersection test
    intersecting = []
    for tile_idx in possible_matches_idx:
        # tile_geom = tiles_gdf.iloc[tile_idx].geometry
        tile_geom = tiles_gdf.iloc[tile_idx].geometry
        if edge_geom.intersects(tile_geom):
            # intersecting.append(tiles_gdf.iloc[tile_idx]['tile_id'])
            intersecting.append(tiles_gdf.iloc[tile_idx]['tile_id'])
    edge_to_tiles[edge_id] = intersecting

# Summary
edge_tile_counts = [len(tlist) for tlist in edge_to_tiles.values()]
print(f"Mapped {len(edge_to_tiles)} edges to tiles")
print(f"Tiles per edge: min={min(edge_tile_counts)}, max={max(edge_tile_counts)}, mean={np.mean(edge_tile_counts):.2f}")
print(f"Edges crossing zero tiles: {sum(1 for c in edge_tile_counts if c == 0)}")

# Preview
print("\nSample edge→tile mappings:")
for edge_id, tile_list in list(edge_to_tiles.items())[:5]:
    print(f"  Edge {edge_id[:30]}... → {len(tile_list)} tiles: {tile_list[:3]}{'...' if len(tile_list)>3 else ''}")

In [ ]:
# Check H_combined date range and filter to study period (2023-03-31 to 2023-04-04)
print("Current H_combined date range:")
print(f"  Start: {H_combined.index.min()}")
print(f"  End: {H_combined.index.max()}")
print(f"  Total time steps: {len(H_combined)}")
print(f"  Timezone: {H_combined.index.tz}")

# Create study period timestamps matching H_combined's timezone
if H_combined.index.tz is None:
    # Timezone-naive
    study_start = pd.Timestamp('2023-03-31')
    study_end = pd.Timestamp('2023-04-04 23:59:59')
else:
    # Timezone-aware
    study_start = pd.Timestamp('2023-03-31', tz=H_combined.index.tz)
    study_end = pd.Timestamp('2023-04-04 23:59:59', tz=H_combined.index.tz)

H_combined_filtered = H_combined[(H_combined.index >= study_start) & (H_combined.index <= study_end)].copy()

print(f"\nFiltered H_combined to study period:")
print(f"  Start: {H_combined_filtered.index.min()}")
print(f"  End: {H_combined_filtered.index.max()}")
print(f"  Time steps: {len(H_combined_filtered)}")
print(f"  Shape: {H_combined_filtered.shape}")

# Replace H_combined with filtered version
H_combined = H_combined_filtered


# Build substation stress vectors

In [ ]:
# Build S_nodes efficiently using vectorized operations
# Strategy: For each hazard, extract all relevant (hazard, tile) columns, 
# then map tile columns to node IDs using a reverse lookup

print("Building S_nodes stress vectors...")
time_index = H_combined.index

# Create reverse mapping: tile_id -> list of node_ids
tile_to_nodes = {}
for node_id, tile_id in node_to_tile.items():
    if tile_id not in tile_to_nodes:
        tile_to_nodes[tile_id] = []
    tile_to_nodes[tile_id].append(node_id)

# Hazard name mapping
hazard_types = {
    'Space Weather (Eh)': 'Eh',
    'Terrestrial Weather (NX3)': 'NX3', 
    'Wildfire (FRP)': 'FRP'
}

# Build S_nodes for each hazard separately, then concatenate
S_nodes_parts = []

for hazard_full_name, hazard_short_name in hazard_types.items():
    # Get all columns for this hazard
    hazard_cols = [col for col in H_combined.columns if col[0] == hazard_full_name]
    
    if len(hazard_cols) == 0:
        print(f"  Warning: No columns found for {hazard_full_name}")
        continue
    
    # Extract hazard data for all tiles
    hazard_data = H_combined[hazard_cols].copy()
    
    # Create node-level dataframe by duplicating tile columns for each node
    node_dfs = []
    for tile_id in tile_to_nodes.keys():
        if (hazard_full_name, tile_id) not in hazard_data.columns:
            continue
        
        tile_series = hazard_data[(hazard_full_name, tile_id)]
        
        # Create a dataframe for all nodes mapped to this tile
        for node_id in tile_to_nodes[tile_id]:
            node_df = pd.DataFrame({
                'time': tile_series.index,
                'node_id': node_id,
                'hazard_type': hazard_short_name,
                'hazard_value': tile_series.values
            })
            node_dfs.append(node_df)
    
    if node_dfs:
        hazard_nodes = pd.concat(node_dfs, ignore_index=True)
        S_nodes_parts.append(hazard_nodes)
        print(f"  {hazard_short_name}: {len(hazard_nodes)} records ({len(tile_to_nodes)} tiles -> {len(node_to_tile)} nodes)")

# Combine all hazards
S_nodes = pd.concat(S_nodes_parts, ignore_index=True)
print(f"\nBuilt S_nodes: {len(S_nodes)} records")
print(f"  Unique nodes: {S_nodes['node_id'].nunique()}")
print(f"  Unique times: {S_nodes['time'].nunique()}")
print(f"  Hazard types: {S_nodes['hazard_type'].unique()}")

# Pivot to wide format
S_nodes_pivot = S_nodes.pivot_table(
    index=['time', 'node_id'], 
    columns='hazard_type', 
    values='hazard_value', 
    aggfunc='first'
)
print(f"\nS_nodes_pivot shape: {S_nodes_pivot.shape}")
S_nodes_pivot.head()

## Save Node/Substation Stress Snapshots

Before attempting edge stress vectors (which may be memory-intensive), save the node-level stress data.

In [ ]:
output_dir = '/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/results/analyses_data/stress_vectors'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Save S_nodes_pivot to parquet (efficient compression for large DataFrames)


# Save the pivot table
output_file_pivot = os.path.join(output_dir, 'S_nodes_pivot_2023_event.parquet')
S_nodes_pivot.to_parquet(output_file_pivot)
print(f"✓ Saved S_nodes_pivot to: {output_file_pivot}")
print(f"  File size: {os.path.getsize(output_file_pivot) / 1e6:.1f} MB")

# Also save the long-form version (smaller, easier to work with)
output_file_long = os.path.join(output_dir, 'S_nodes_long_2023_event.parquet')
S_nodes.to_parquet(output_file_long)
print(f"✓ Saved S_nodes (long form) to: {output_file_long}")
print(f"  File size: {os.path.getsize(output_file_long) / 1e6:.1f} MB")

# Save metadata
import json
metadata = {
    'event_period': '2023-03-31 to 2023-04-04',
    'num_nodes': int(S_nodes_pivot.index.get_level_values('node_id').nunique()),
    'num_timesteps': int(S_nodes_pivot.index.get_level_values('time').nunique()),
    'hazard_types': list(S_nodes_pivot.columns),
    'shape': list(S_nodes_pivot.shape),
    'memory_gb': float(S_nodes_pivot.memory_usage(deep=True).sum() / 1e9),
    'created': pd.Timestamp.now().isoformat()
}
metadata_file = os.path.join(output_dir, 'S_nodes_metadata.json')
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✓ Saved metadata to: {metadata_file}")

# Resume Workflow: Reload Saved Node Stress & Build Sparse Representation

This section lets you pick up analysis without regenerating `S_nodes_pivot`. It loads the saved parquet files, validates integrity, and (optionally) creates a sparse, event-only node stress table to reduce memory footprint.

In [ ]:

def load_saved_node_stress(base_dir='/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/results/analyses_data/stress_vectors/', rebuild_sparse=True, save_sparse=True):
    """Load previously saved node stress data and optionally build a sparse representation.
    Returns a dict with keys: pivot, long, meta, sparse (optional)."""
    pivot_path = os.path.join(base_dir, 'S_nodes_pivot_2023_event.parquet')
    long_path = os.path.join(base_dir, 'S_nodes_long_2023_event.parquet')
    meta_path = os.path.join(base_dir, 'S_nodes_metadata.json')
    sparse_path = os.path.join(base_dir, 'S_nodes_sparse_2023_event.parquet')
    meta_sparse_path = os.path.join(base_dir, 'S_nodes_sparse_metadata.json')
    out = {}

    if os.path.exists(pivot_path):
        pivot_df = pd.read_parquet(pivot_path)
        out['pivot'] = pivot_df
        print(f"Loaded pivot: {pivot_path} -> shape={pivot_df.shape}, mem={pivot_df.memory_usage(deep=True).sum()/1e9:.2f} GB")
    else:
        print("Pivot parquet not found; regenerate before continuing.")

    if os.path.exists(long_path):
        long_df = pd.read_parquet(long_path)
        out['long'] = long_df
        print(f"Loaded long:  {long_path} -> rows={len(long_df):,}, mem={long_df.memory_usage(deep=True).sum()/1e6:.1f} MB")
    else:
        print("Long-form parquet not found.")

    if os.path.exists(meta_path):
        with open(meta_path) as f:
            out['meta'] = json.load(f)
        print("Loaded metadata.")

    # Sparse: build only if requested and not existing already
    need_build = rebuild_sparse or (not os.path.exists(sparse_path))
    if need_build and 'pivot' in out:
        pivot_df = out['pivot']
        # Ensure MultiIndex properly named
        if pivot_df.index.names != ['time','node_id']:
            # Attempt to infer if index lost; rebuild from long if possible
            if 'long' in out:
                pivot_df = out['long'].pivot_table(index=['time','node_id'], columns='hazard_type', values='hazard_value', aggfunc='first')
                out['pivot'] = pivot_df
                print("Reconstructed pivot index from long form.")
            else:
                print("Warning: pivot index names unexpected and no long form to reconstruct.")
        
        records = []
        # Iterate hazard columns one at a time to keep memory lower
        for hazard in pivot_df.columns:
            col = pivot_df[hazard]
            # Select non-zero & non-NaN
            nz = col[(col.notna()) & (col != 0)]
            if nz.empty:
                continue
            idx = nz.index
            records.append(pd.DataFrame({
                'time': idx.get_level_values('time'),
                'node_id': idx.get_level_values('node_id'),
                'hazard_type': hazard,
                'hazard_value': nz.values
            }))
        if records:
            sparse_df = pd.concat(records, ignore_index=True)
        else:
            sparse_df = pd.DataFrame(columns=['time','node_id','hazard_type','hazard_value'])
        out['sparse'] = sparse_df
        print(f"Built sparse node stress: {len(sparse_df):,} rows (compression ratio ≈ {(len(pivot_df)*len(pivot_df.columns))/max(len(sparse_df),1):.1f}x)")
        print(f"Sparse memory: {sparse_df.memory_usage(deep=True).sum()/1e6:.1f} MB")

        if save_sparse:
            os.makedirs(base_dir, exist_ok=True)
            sparse_df.to_parquet(sparse_path)
            meta_sparse = {
                'source_pivot_shape': list(pivot_df.shape),
                'sparse_rows': int(len(sparse_df)),
                'hazards': list(pivot_df.columns),
                'compression_ratio': float((len(pivot_df)*len(pivot_df.columns))/max(len(sparse_df),1)),
                'memory_mb': float(sparse_df.memory_usage(deep=True).sum()/1e6),
                'created': pd.Timestamp.now().isoformat()
            }
            with open(meta_sparse_path,'w') as f:
                json.dump(meta_sparse, f, indent=2)
            print(f"Saved sparse + metadata to disk.")
    else:
        if os.path.exists(sparse_path):
            sparse_df = pd.read_parquet(sparse_path)
            out['sparse'] = sparse_df
            print(f"Loaded existing sparse node stress: {sparse_path} ({len(sparse_df):,} rows)")
            if os.path.exists(meta_sparse_path):
                with open(meta_sparse_path) as f:
                    out['sparse_meta'] = json.load(f)
        else:
            print("Sparse node stress not built or requested.")

    return out

# Helper to reconstruct a single node's full hazard time series from sparse if pivot absent
def reconstruct_node_timeseries_from_sparse(node_id, sparse_df, time_index=None):
    if time_index is None:
        # Try to infer from pivot or metadata
        if 'pivot' in globals() and isinstance(S_nodes_pivot, pd.DataFrame):
            time_index = S_nodes_pivot.index.get_level_values('time').unique()
        else:
            time_index = sparse_df['time'].sort_values().unique()
    node_sparse = sparse_df[sparse_df['node_id'] == node_id]
    if node_sparse.empty:
        return pd.DataFrame(index=time_index, columns=sparse_df['hazard_type'].unique())
    pivot = node_sparse.pivot_table(index='time', columns='hazard_type', values='hazard_value', aggfunc='first')
    pivot = pivot.reindex(time_index)
    return pivot

# Execute loader
node_assets = load_saved_node_stress(base_dir='/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/results/analyses_data/stress_vectors/',rebuild_sparse=True, save_sparse=True)

# Make globals convenient if not already present
if 'pivot' in node_assets and 'S_nodes_pivot' not in globals():
    S_nodes_pivot = node_assets['pivot']
if 'sparse' in node_assets:
    S_nodes_sparse = node_assets['sparse']

print("\nResume node stress loading complete.")

if 'flag_node_sparsification' in locals(): 
    del flag_node_sparsification

In [ ]:
# Fix the sparse DataFrame memory issue (if desired)
# This converts string columns to categorical, reducing memory ~40x


if 'S_nodes_sparse' in locals() and 'flag_node_sparsification' not in locals():
    print("FIXING S_nodes_sparse memory issue...")
    print(f"Before: {S_nodes_sparse.memory_usage(deep=True).sum() / 1e9:.2f} GB")
    
    # Convert to categorical
    S_nodes_sparse['node_id'] = S_nodes_sparse['node_id'].astype('category')
    S_nodes_sparse['hazard_type'] = S_nodes_sparse['hazard_type'].astype('category')
    
    print(f"After:  {S_nodes_sparse.memory_usage(deep=True).sum() / 1e6:.1f} MB")
    print(f"\nReduction: {12800 / (S_nodes_sparse.memory_usage(deep=True).sum() / 1e6):.1f}x smaller!")
    
    # Optionally save the fixed version
    # S_nodes_sparse.to_parquet(os.path.join(base_dir, 'S_nodes_sparse_2023_event_FIXED.parquet'))
else:
    print("S_nodes_sparse not loaded - nothing to fix")
    
flag_node_sparsification = 1

# Build transmission line stress vector



## Full Edge Stress analysis - currently computationally infeasible


In [ ]:
# Hazard name mapping
hazard_types = {
    'Space Weather (Eh)': 'Eh',
    'Terrestrial Weather (NX3)': 'NX3', 
    'Wildfire (FRP)': 'FRP'
}

In [ ]:
# Estimate S_edges size before building
num_edges = len(edge_to_tiles)
num_hazards = len(hazard_types)
num_times = len(H_combined)

estimated_records = num_edges * num_hazards * num_times
estimated_memory_gb = estimated_records * 8 * 4 / 1e9  # 4 columns × 8 bytes each

print(f"Edge stress vector estimates:")
print(f"  Edges: {num_edges:,}")
print(f"  Hazards: {num_hazards}")
print(f"  Time steps: {num_times}")
print(f"  Estimated records: {estimated_records:,}")
print(f"  Estimated memory: {estimated_memory_gb:.2f} GB")
print(f"\nFor comparison:")
print(f"  S_nodes actual: {S_nodes_pivot.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"  Ratio (edges/nodes): {num_edges / len(node_to_tile):.1f}x")




In [3]:
print('NOTE, the full edge stress generation code is not included here - could be created by implementing the same workflow that we have for nodes')

NOTE, the full edge stress generation code is not included here - could be created by implementing the same workflow that we have for nodes


## Build transmission line/edge stress vectors with sparse representation or for 'critical' edges only

In [ ]:
# Build S_edges using SPARSE representation with chunked processing
# Only store non-zero/non-NaN stress values to reduce memory

print("Building S_edges with sparse representation...")
print(f"Processing {len(edge_to_tiles)} edges in chunks...")

# Process in chunks to avoid memory issues
chunk_size = 1000  # edges per chunk
edge_items = list(edge_to_tiles.items())
num_chunks = (len(edge_items) + chunk_size - 1) // chunk_size

# We'll save chunks to disk incrementally
sparse_records = []
edges_processed = 0

for chunk_idx in range(num_chunks):
    start_idx = chunk_idx * chunk_size
    end_idx = min(start_idx + chunk_size, len(edge_items))
    chunk_edges = edge_items[start_idx:end_idx]
    
    print(f"\nChunk {chunk_idx + 1}/{num_chunks}: edges {start_idx}-{end_idx}")
    
    chunk_records = []
    
    for hazard_full_name, hazard_short_name in hazard_types.items():
        # Get all columns for this hazard
        hazard_cols = [col for col in H_combined.columns if col[0] == hazard_full_name]
        
        if len(hazard_cols) == 0:
            continue
        
        hazard_data = H_combined[hazard_cols].copy()
        
        for edge_id, tile_list in chunk_edges:
            if len(tile_list) == 0:
                continue
            
            # Get columns for tiles this edge crosses
            relevant_cols = [(hazard_full_name, tid) for tid in tile_list 
                            if (hazard_full_name, tid) in hazard_data.columns]
            
            if len(relevant_cols) == 0:
                continue
            
            # Get max across tiles for each timestep
            if len(relevant_cols) == 1:
                max_series = hazard_data[relevant_cols[0]]
            else:
                max_series = hazard_data[relevant_cols].max(axis=1)
            
            # SPARSE: Only store non-zero and non-NaN values
            for t, val in max_series.items():
                if pd.notna(val) and val != 0:
                    chunk_records.append({
                        'time': t,
                        'edge_id': edge_id,
                        'hazard_type': hazard_short_name,
                        'hazard_value': val,
                        'num_tiles': len(tile_list)
                    })
    
    edges_processed += len(chunk_edges)
    sparse_records.extend(chunk_records)
    
    print(f"  Processed: {len(chunk_edges)} edges")
    print(f"  Sparse records in chunk: {len(chunk_records)}")
    print(f"  Total sparse records: {len(sparse_records)}")
    print(f"  Memory estimate: {len(sparse_records) * 40 / 1e6:.1f} MB")

print(f"\n✓ Built sparse S_edges: {len(sparse_records)} records (non-zero only)")
print(f"  Total edges processed: {edges_processed}")
print(f"  Compression ratio: {(edges_processed * len(hazard_types) * len(H_combined)) / max(len(sparse_records), 1):.1f}x")

# Convert to DataFrame
S_edges_sparse = pd.DataFrame(sparse_records)
print(f"\nS_edges_sparse shape: {S_edges_sparse.shape}")
print(f"Memory: {S_edges_sparse.memory_usage(deep=True).sum() / 1e6:.1f} MB")
S_edges_sparse.head()

In [ ]:
# Save sparse edge stress data
output_file_sparse = os.path.join(output_dir, 'S_edges_sparse_2023_event.parquet')
S_edges_sparse.to_parquet(output_file_sparse)
print(f"✓ Saved S_edges_sparse to: {output_file_sparse}")
print(f"  File size: {os.path.getsize(output_file_sparse) / 1e6:.1f} MB")

# Save metadata
edge_metadata = {
    'event_period': '2023-03-31 to 2023-04-04',
    'num_edges': int(S_edges_sparse['edge_id'].nunique()),
    'num_timesteps': int(S_edges_sparse['time'].nunique()),
    'hazard_types': list(S_edges_sparse['hazard_type'].unique()),
    'sparse_records': len(S_edges_sparse),
    'representation': 'sparse (non-zero only)',
    'memory_mb': float(S_edges_sparse.memory_usage(deep=True).sum() / 1e6),
    'created': pd.Timestamp.now().isoformat()
}
edge_metadata_file = os.path.join(output_dir, 'S_edges_sparse_metadata.json')
with open(edge_metadata_file, 'w') as f:
    json.dump(edge_metadata, f, indent=2)
print(f"✓ Saved metadata to: {edge_metadata_file}")

In [ ]:
# Strategy 1: Identify critical edges based on network topology
def get_critical_edges_topological(top_n_percent=10):
    """
    Select edges connecting high-degree nodes (critical backbone).
    
    Returns: dict with edge_id -> importance_score
    """
    # Get node degrees
    node_degrees = dict(G.degree())
    
    # Score edges by sum of endpoint degrees
    edge_importance = {}
    for edge_id, row in edges_gdf_full.iterrows():
        u = row.get('start_node', row.get('u', None))
        v = row.get('end_node', row.get('v', None))
        
        if u and v and u in node_degrees and v in node_degrees:
            # Importance = sum of degrees (backbone edges)
            importance = node_degrees[u] + node_degrees[v]
            edge_importance[edge_id] = importance
    
    # Get top N%
    n_top = int(len(edge_importance) * top_n_percent / 100)
    top_edges = sorted(edge_importance.items(), key=lambda x: x[1], reverse=True)[:n_top]
    
    print(f"Strategy 1: Topological Criticality")
    print(f"  Total edges: {len(edge_importance)}")
    print(f"  Selected top {top_n_percent}%: {len(top_edges)} edges")
    print(f"  Importance range: {top_edges[-1][1]:.0f} to {top_edges[0][1]:.0f}")
    
    return dict(top_edges)

# Strategy 2: Identify edges crossing hazardous tiles
def get_critical_edges_hazard_exposure(min_tiles=1):
    """
    Select edges that cross tiles with significant hazard activity.
    
    Returns: dict with edge_id -> num_hazardous_tiles
    """
    # Identify hazardous tiles (tiles with any stress > threshold)
    hazard_threshold = 0.1  # adjust based on hazard type
    hazardous_tiles = set()
    
    for col in H_combined.columns:
        hazard_type, tile_id = col
        if H_combined[col].max() > hazard_threshold:
            hazardous_tiles.add(tile_id)
    
    print(f"\nStrategy 2: Hazard Exposure")
    print(f"  Hazardous tiles (max stress > {hazard_threshold}): {len(hazardous_tiles)}")
    
    # Select edges crossing hazardous tiles
    edge_exposure = {}
    for edge_id, tile_list in edge_to_tiles.items():
        n_hazardous = len([t for t in tile_list if t in hazardous_tiles])
        if n_hazardous >= min_tiles:
            edge_exposure[edge_id] = n_hazardous
    
    print(f"  Edges crossing ≥{min_tiles} hazardous tiles: {len(edge_exposure)}")
    
    return edge_exposure

# Strategy 3: Combined scoring (topology + exposure)
def get_critical_edges_combined(top_n=1000):
    """
    Combine topological importance and hazard exposure.
    
    Returns: list of top N edge_ids
    """
    topo = get_critical_edges_topological(100)  # get all scores
    expo = get_critical_edges_hazard_exposure(0)  # get all exposures
    
    # Normalize scores to 0-1
    if topo:
        max_topo = max(topo.values())
        topo_norm = {k: v/max_topo for k, v in topo.items()}
    else:
        topo_norm = {}
    
    if expo:
        max_expo = max(expo.values())
        expo_norm = {k: v/max_expo for k, v in expo.items()}
    else:
        expo_norm = {}
    
    # Combined score (weighted average)
    combined = {}
    all_edges = set(topo_norm.keys()) | set(expo_norm.keys())
    
    for edge_id in all_edges:
        t_score = topo_norm.get(edge_id, 0)
        e_score = expo_norm.get(edge_id, 0)
        # Weight: 60% topology, 40% exposure
        combined[edge_id] = 0.6 * t_score + 0.4 * e_score
    
    # Get top N
    top_edges = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:top_n]
    
    print(f"\nStrategy 3: Combined (Topology + Exposure)")
    print(f"  Selected top {top_n} edges")
    print(f"  Score range: {top_edges[-1][1]:.3f} to {top_edges[0][1]:.3f}")
    
    return [edge_id for edge_id, score in top_edges]

# Execute strategies
critical_topo = get_critical_edges_topological(top_n_percent=10)
critical_expo = get_critical_edges_hazard_exposure(min_tiles=1)
critical_combined = get_critical_edges_combined(top_n=1000)

In [ ]:
# Build sparse edge stress for SELECTED edges only
def build_sparse_edge_stress_subset(edge_subset, chunk_size=200):
    """
    Build sparse edge stress for a subset of edges.
    
    Parameters:
    - edge_subset: list of edge_ids to process
    - chunk_size: edges per chunk
    
    Returns: DataFrame with sparse edge stress records
    """
    print(f"Building sparse edge stress for {len(edge_subset)} selected edges...")
    
    # Filter edge_to_tiles to selected edges
    selected_edge_tiles = {eid: edge_to_tiles[eid] for eid in edge_subset if eid in edge_to_tiles}
    
    edge_items = list(selected_edge_tiles.items())
    num_chunks = (len(edge_items) + chunk_size - 1) // chunk_size
    
    sparse_records = []
    
    for chunk_idx in range(num_chunks):
        start_idx = chunk_idx * chunk_size
        end_idx = min(start_idx + chunk_size, len(edge_items))
        chunk_edges = edge_items[start_idx:end_idx]
        
        if (chunk_idx + 1) % 5 == 0 or chunk_idx == 0:
            print(f"  Chunk {chunk_idx + 1}/{num_chunks}: edges {start_idx}-{end_idx}")
        
        for hazard_full_name, hazard_short_name in hazard_types.items():
            hazard_cols = [col for col in H_combined.columns if col[0] == hazard_full_name]
            
            if len(hazard_cols) == 0:
                continue
            
            hazard_data = H_combined[hazard_cols].copy()
            
            for edge_id, tile_list in chunk_edges:
                if len(tile_list) == 0:
                    continue
                
                relevant_cols = [(hazard_full_name, tid) for tid in tile_list 
                                if (hazard_full_name, tid) in hazard_data.columns]
                
                if len(relevant_cols) == 0:
                    continue
                
                # Get max across tiles
                if len(relevant_cols) == 1:
                    max_series = hazard_data[relevant_cols[0]]
                else:
                    max_series = hazard_data[relevant_cols].max(axis=1)
                
                # SPARSE: only non-zero, non-NaN
                for t, val in max_series.items():
                    if pd.notna(val) and val != 0:
                        sparse_records.append({
                            'time': t,
                            'edge_id': edge_id,
                            'hazard_type': hazard_short_name,
                            'hazard_value': val,
                            'num_tiles': len(tile_list)
                        })
    
    df = pd.DataFrame(sparse_records)
    print(f"\n✓ Built sparse edge stress: {len(df)} records")
    print(f"  Edges with stress: {df['edge_id'].nunique()}")
    print(f"  Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
    
    return df

# Build stress for top 1000 critical edges (combined strategy)
print("Building edge stress for critical subset...")
S_edges_critical = build_sparse_edge_stress_subset(critical_combined, chunk_size=200)

In [ ]:
# Save critical edge stress data
output_file_critical = os.path.join(output_dir, 'S_edges_critical_sparse_2023_event.parquet')
S_edges_critical.to_parquet(output_file_critical)
print(f"✓ Saved critical edge stress to: {output_file_critical}")
print(f"  File size: {os.path.getsize(output_file_critical) / 1e6:.1f} MB")

# Save metadata about subsetting strategy
critical_edge_meta = {
    'event_period': '2023-03-31 to 2023-04-04',
    'total_edges_in_network': len(edge_to_tiles),
    'critical_edges_selected': len(critical_combined),
    'selection_strategy': 'combined_topology_exposure',
    'topology_weight': 0.6,
    'exposure_weight': 0.4,
    'sparse_records': len(S_edges_critical),
    'edges_with_stress': int(S_edges_critical['edge_id'].nunique()),
    'hazard_types': list(S_edges_critical['hazard_type'].unique()),
    'memory_mb': float(S_edges_critical.memory_usage(deep=True).sum() / 1e6),
    'created': pd.Timestamp.now().isoformat()
}

meta_file = os.path.join(output_dir, 'S_edges_critical_metadata.json')
with open(meta_file, 'w') as f:
    json.dump(critical_edge_meta, f, indent=2)
print(f"✓ Saved metadata to: {meta_file}")

In [ ]:
# Summary of component-hazard mapping
print("=== Component Stress State Summary ===\n")
print(f"S_nodes_pivot: {S_nodes_pivot.shape[0]} (time×node) observations")
print(f"  Unique nodes: {len(S_nodes['node_id'].unique())}")
print(f"  Unique times: {len(S_nodes['time'].unique())}")
print(f"  Hazard types: {list(S_nodes_pivot.columns)}")
print(f"  Coverage (non-NaN): {S_nodes_pivot.notna().sum().to_dict()}")

print(f"\nS_edges_critical: {S_edges_critical.shape[0]} (time×edge) observations")
print(f"  Unique edges: {len(S_edges_critical['edge_id'].unique())}")
print(f"  Unique times: {len(S_edges_critical['time'].unique())}")
print(f"  Hazard types: {list(S_edges_critical.columns)}")
print(f"  Coverage (non-NaN): {S_edges_critical.notna().sum().to_dict()}")

print("\nYou can now:")
print("  • Analyze temporal hazard exposure for individual components")
print("  • Identify critical nodes/edges with persistent high hazard")
print("  • Model cascading failures based on component stress")
print("  • Compute vulnerability scores combining multiple hazards")
print("  • Correlate network topology (degree, betweenness) with hazard exposure")

# Multi-Hazard Impact Analysis: Copula-Based Joint Tail Extraction & Impact Functions

This section integrates EAGLE-I outage data with the stress vectors to pursue many possible objectives, including:

**Goal: Copula-Based Joint Hazard Tail Extraction**  
Understand how joint hazard tail events (multi-hazard co-occurrence) differ from individual hazard extremes. Use copula models to capture tail dependence structure and identify critical joint thresholds.

**Goal: Multi-Hazard Impact Function Quantification**  
Establish dose-response relationships between stress vectors and grid outages: P(outage | stress_Eh, stress_NX3, stress_FRP, topology). Quantify single-hazard vs multi-hazard predictive power and identify synergy effects.

---

**Prerequisites:** Before running this section, ensure you've executed earlier cells that create:
- `G` (NetworkX graph) - created in cell 16
- `nodes_gdf` (GeoDataFrame with node geometries) - needs CRS and columns added
- `edges_gdf` (GeoDataFrame with edge geometries)
- `top_X_nodes` (list of node IDs in subgraph) 
- `S_nodes_pivot` (stress vectors) - loaded earlier (Or `S_nodes_sparse`)

## Load outage data

In [ ]:
# Load EAGLE-I outage data for 2023 event
 
# Path to EAGLE-I data
eaglei_directory = '/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/outage_data/EAGLE-I/'
eaglei_file = 'outage_data_2023.csv'

print(f"Loading EAGLE-I data from {os.path.join(eaglei_directory, eaglei_file)}...")
outage_data = pd.read_csv(os.path.join(eaglei_directory, eaglei_file))

# Convert timestamp to datetime
outage_data['datetime'] = pd.to_datetime(outage_data['run_start_time'])

# Filter to study period (March 31 - April 4, 2023)
study_start = '2023-03-31'
study_end = '2023-04-04'
outage_filtered = outage_data[(outage_data['datetime'] >= study_start) & 
                               (outage_data['datetime'] <= study_end)].copy()

print(f"\nOriginal EAGLE-I data shape: {outage_data.shape}")
print(f"Filtered to study period: {outage_filtered.shape}")
print(f"\nColumns: {list(outage_filtered.columns)}")
print(f"\nDate range: {outage_filtered['datetime'].min()} to {outage_filtered['datetime'].max()}")
print(f"Temporal resolution: ~{(outage_filtered['datetime'].diff().max())}")
print(f"\nSample data:")
print(outage_filtered.head())

In [ ]:
# Explore outage statistics
print("Outage Statistics for Study Period:")
print(f"Total records: {len(outage_filtered)}")
print(f"Unique counties: {outage_filtered['county'].nunique()}")
print(f"Unique states: {outage_filtered['state'].nunique()}")
print(f"\nOutage summary (customers without power):")
print(outage_filtered['sum'].describe())
print(f"\nCounties with any outages (Sum > 0): {(outage_filtered['sum'] > 0).sum()} records")
print(f"Percentage of records with outages: {100 * (outage_filtered['sum'] > 0).sum() / len(outage_filtered):.2f}%")

In [ ]:
# Resample to hourly resolution

print("\nResampling outage data to hourly resolution...")
# Group by county and hour, sum customers without power
outage_filtered['datetime_hour'] = outage_filtered['datetime'].dt.floor('H')
outage_hourly = outage_filtered.groupby(['fips_code', 'datetime_hour'])['sum'].sum().reset_index()
outage_hourly.rename(columns={'datetime_hour': 'datetime', 'sum': 'customers_out'}, inplace=True)
print(f"outage_hourly shape: {outage_hourly.shape}")
print(f"\nSample hourly outage data:")
print(outage_hourly.head())

## Load County Geometries and Spatial Join to Grid Nodes

In [ ]:
# Load NERC regions
regions_gdf = gpd.read_file("/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/NERC_Reliability_Coordinators.geojson")
# Ensure the CRS is WGS84 (EPSG:4326)
if regions_gdf.crs is None or regions_gdf.crs.to_string() != "EPSG:4326":
    print('converting to EPSG: 4326')
    regions_gdf = regions_gdf.to_crs("EPSG:4326")

# Load the US County GeoJSON file
counties_url = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
print(f"Downloading county geometries from {counties_url}...")
response = requests.get(counties_url)
counties_geojson = response.json()


# Convert to GeoDataFrame
counties_records = []
for feature in counties_geojson['features']:
    geom = shape(feature['geometry'])
    fips = feature['id']
    counties_records.append({
        'fips_code': fips,
        'geometry': geom
    })

counties_gdf = gpd.GeoDataFrame(counties_records, crs='EPSG:4326')
print(f"\nCounties GeoDataFrame shape: {counties_gdf.shape}")
print(f"CRS: {counties_gdf.crs}")
print(counties_gdf.head())

In [ ]:
# Ensure EAGLE-I data has FIPS codes formatted correctly (5-digit string with leading zeros)
outage_filtered['fips_code'] = outage_filtered['fips_code'].astype(str).str.zfill(5)

# Merge outage data with county geometries
outage_with_geom = counties_gdf.merge(outage_filtered, on='fips_code', how='inner')
outage_with_geom = gpd.GeoDataFrame(outage_with_geom, crs='EPSG:4326')

print(f"\nOutage data merged with county geometries: {outage_with_geom.shape}")
print(f"Records with geometry: {outage_with_geom['geometry'].notna().sum()}")
print(f"\nSample merged data:")
print(outage_with_geom[['fips_code', 'county', 'state', 'datetime', 'sum', 'geometry']].head())

In [ ]:
# Spatial join: determine which grid nodes fall within which counties
# This creates a mapping from nodes to counties
print("Performing spatial join: nodes → counties...")
print(f"nodes_gdf CRS: {nodes_gdf.crs}")
print(f"counties_gdf CRS: {counties_gdf.crs}")

# Ensure both are in same CRS (should already be EPSG:4326)
if nodes_gdf.crs != counties_gdf.crs:
    print("CRS mismatch - reprojecting...")
    nodes_gdf = nodes_gdf.to_crs(counties_gdf.crs)

# Spatial join: find which county each node falls within
nodes_with_counties = gpd.sjoin(
    nodes_gdf, 
    counties_gdf[['fips_code', 'geometry']], 
    how='left', 
    predicate='within'
)

print(f"\nSpatial join complete.")
print(f"Nodes with county assignments: {nodes_with_counties['fips_code'].notna().sum()} / {len(nodes_with_counties)}")
print(f"Nodes without county (likely offshore/border issues): {nodes_with_counties['fips_code'].isna().sum()}")
print(f"\nSample node-county mapping:")
print(nodes_with_counties[['node_id', 'fips_code', 'degree', 'geometry']].head(10))

In [ ]:
# ISO/NERC region outages: spatially join counties to regions and aggregate hourly

# Ensure both GeoDataFrames are in the same CRS
if counties_gdf.crs is None:
    counties_gdf = counties_gdf.set_crs('EPSG:4326')
if regions_gdf.crs is None:
    regions_gdf = regions_gdf.set_crs('EPSG:4326')
if counties_gdf.crs != regions_gdf.crs:
    counties_gdf = counties_gdf.to_crs(regions_gdf.crs)

# Use NAME column for NERC region identifiers
region_name_col = 'NAME'
print(f"Using region name column: '{region_name_col}'")
print(f"Available NERC regions: {sorted(regions_gdf[region_name_col].unique().tolist())}")

# Ensure FIPS are 5-digit strings (counties_gdf uses 'id' column)
counties_gdf['fips_code'] = counties_gdf['id'].astype(str).str.zfill(5)
outage_hourly['fips_code'] = outage_hourly['fips_code'].astype(str).str.zfill(5)

# Try strict containment first
counties_regions_within = gpd.sjoin(
    counties_gdf[['fips_code','geometry']],
    regions_gdf[[region_name_col,'geometry']],
    predicate='within',
    how='left'
)

if counties_regions_within[region_name_col].isna().any():
    # Fallback: intersects, then pick region with largest overlap area
    intersect = gpd.sjoin(
        counties_gdf[['fips_code','geometry']],
        regions_gdf[[region_name_col,'geometry']],
        predicate='intersects',
        how='left'
    )

    # Ensure we have a valid index to reference the right-side geometry
    # index_right is added by sjoin and corresponds to regions_gdf.index
    def compute_overlap(r):
        # No region matched
        if pd.isna(r.get('index_right')):
            return 0.0
        # Some rows can have invalid geometries; buffer(0) can fix simple issues
        county_geom = r['geometry']
        region_geom = regions_gdf.loc[r['index_right'], 'geometry']
        if county_geom is None or region_geom is None:
            return 0.0
        try:
            return county_geom.intersection(region_geom).area
        except Exception:
            return county_geom.buffer(0).intersection(region_geom.buffer(0)).area

    intersect['overlap_area'] = intersect.apply(compute_overlap, axis=1)

    # Choose region with max overlap per county
    counties_regions = (
        intersect
        .sort_values(['fips_code', 'overlap_area'], ascending=[True, False])
        .drop_duplicates('fips_code')
        [['fips_code', region_name_col]]
        .reset_index(drop=True)
    )
else:
    counties_regions = counties_regions_within[['fips_code', region_name_col]].copy()


# Build a unique mapping
county_to_region = (
    counties_regions
    .dropna(subset=[region_name_col])
    .drop_duplicates(subset=['fips_code'])
    .set_index('fips_code')[region_name_col]
)

# Map region to outages and aggregate
outage_hourly['region'] = outage_hourly['fips_code'].map(county_to_region)
out_col = 'customers_out'
region_hourly = (
    outage_hourly.dropna(subset=['region'])
    .groupby(['datetime','region'], as_index=False)[out_col]
    .sum()
)

print(f"NERC regions with outage data: {sorted(region_hourly['region'].unique())}")


In [ ]:
# Build integrated panel: (time, node_id) with stress + outage columns
# Strategy: 
# 1. Reshape S_nodes_hourly from wide (multi-index columns) to long format
# 2. Add county FIPS code to each node
# 3. Merge with hourly outage data

print("Building integrated panel dataset...")

# Reshape stress data to long format
# Currently: columns are (hazard_type, node_id)
# Want: columns are [time, node_id, Eh, NX3, FRP]

# Extract each hazard type separately
stress_eh = S_nodes_hourly['Eh'].copy()
stress_nx3 = S_nodes_hourly['NX3'].copy()
stress_frp = S_nodes_hourly['FRP'].copy()

# Stack each to create long format: (time, node_id) index with stress value
stress_eh_long = stress_eh.reset_index()
stress_eh_long.columns = ['node_id', 'datetime', 'stress_Eh']

stress_nx3_long = stress_nx3.reset_index()
stress_nx3_long.columns = ['node_id', 'datetime', 'stress_NX3']

stress_frp_long = stress_frp.reset_index()
stress_frp_long.columns = ['node_id', 'datetime', 'stress_FRP']

# Merge all hazards
panel = stress_eh_long.merge(stress_nx3_long, on=['datetime', 'node_id'], how='outer')
panel = panel.merge(stress_frp_long, on=['datetime', 'node_id'], how='outer')

print(f"Panel with stress data: {panel.shape}")
print(f"Unique nodes: {panel['node_id'].nunique()}")
print(f"Unique timestamps: {panel['datetime'].nunique()}")
print(f"\nSample panel:")
print(panel.head())

In [ ]:
# Add county FIPS codes to panel
panel['fips_code'] = panel['node_id'].map(node_to_county)

# Normalize FIPS type to 5-char zero-padded string on both dataframes
panel['fips_code'] = panel['fips_code'].astype(str).str.zfill(5)
outage_hourly['fips_code'] = outage_hourly['fips_code'].astype(str).str.zfill(5)

# Ensure datetime columns are proper datetimes
panel['datetime'] = pd.to_datetime(panel['datetime'])
outage_hourly['datetime'] = pd.to_datetime(outage_hourly['datetime'])

# Merge with outage data
panel = panel.merge(outage_hourly, on=['fips_code', 'datetime'], how='left')

# Fill NaN outages with 0 (no reported outage)
panel['customers_out'] = panel['customers_out'].fillna(0)

# Create binary outage flag
panel['outage_flag'] = (panel['customers_out'] > 0).astype(int)

# Add topology metrics (degree from nodes_gdf)
node_degree = nodes_gdf.set_index('node_id')['degree'].to_dict()
panel['degree'] = panel['node_id'].map(node_degree)

print(f"\nIntegrated panel dataset complete!")
print(f"Shape: {panel.shape}")
print(f"Columns: {list(panel.columns)}")
print(f"\nMissing values:")
print(panel.isnull().sum())
print(f"\nOutage statistics in panel:")
print(f"Records with outages: {panel['outage_flag'].sum()} / {len(panel)}")
print(f"Percentage: {100 * panel['outage_flag'].sum() / len(panel):.2f}%")
print(f"\nSample integrated panel:")
print(panel[panel['outage_flag'] == 1].head(10))

In [ ]:
# Save integrated panel for future use
output_dir = '/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/results/impact_analysis/'
os.makedirs(output_dir, exist_ok=True)

panel_file = os.path.join(output_dir, 'stress_outage_panel_hourly_2023_event.parquet')
panel.to_parquet(panel_file, compression='gzip')
print(f"\nIntegrated panel saved to: {panel_file}")
print(f"File size: {os.path.getsize(panel_file) / 1024**2:.2f} MB")

## Outage time series: overall, by state, and by ISO/NERC region


In [ ]:

# Plot
pivot_regions = region_hourly.pivot(index='datetime', columns='region', values=out_col).fillna(0)
fig, ax = plt.subplots(figsize=(14, 6))
pivot_regions.plot(ax=ax, lw=1.5, marker='o', markersize=2)
ax.set_title('Hourly outages by NERC region', fontsize=14, fontweight='bold')
ax.set_xlabel('Time')
ax.set_ylabel('Customers out')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), title='NERC Region', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
State	Postal
Abbr.	FIPS
Code	State	Postal
Abbr.	FIPS
Code
Alabama	AL	01	Nebraska	NE	31
Alaska	AK	02	Nevada	NV	32
Arizona	AZ	04	New Hampshire	NH	33
Arkansas	AR	05	New Jersey	NJ	34
California	CA	06	New Mexico	NM	35
Colorado	CO	08	New York	NY	36
Connecticut	CT	09	North Carolina	NC	37
Delaware	DE	10	North Dakota	ND	38
District of Columbia	DC	11	Ohio	OH	39
Florida	FL	12	Oklahoma	OK	40
Georgia	GA	13	Oregon	OR	41
Hawaii	HI	15	Pennsylvania	PA	42
Idaho	ID	16	Puerto Rico	PR	72
Illinois	IL	17	Rhode Island	RI	44
Indiana	IN	18	South Carolina	SC	45
Iowa	IA	19	South Dakota	SD	46
Kansas	KS	20	Tennessee	TN	47
Kentucky	KY	21	Texas	TX	48
Louisiana	LA	22	Utah	UT	49
Maine	ME	23	Vermont	VT	50
Maryland	MD	24	Virginia	VA	51
Massachusetts	MA	25	Virgin Islands	VI	78
Michigan	MI	26	Washington	WA	53
Minnesota	MN	27	West Virginia	WV	54
Mississippi	MS	28	Wisconsin	WI	55
Missouri	MO	29	Wyoming	WY	56
Montana	MT	30	 

In [ ]:
# State-by-state outages: map two-digit state FIPS codes to state names
# Using the reference table from the cell above

# FIPS code to state name mapping (from reference table above)
fips_to_state = {
    '01': 'Alabama', '02': 'Alaska', '04': 'Arizona', '05': 'Arkansas', '06': 'California',
    '08': 'Colorado', '09': 'Connecticut', '10': 'Delaware', '11': 'District of Columbia', '12': 'Florida',
    '13': 'Georgia', '15': 'Hawaii', '16': 'Idaho', '17': 'Illinois', '18': 'Indiana',
    '19': 'Iowa', '20': 'Kansas', '21': 'Kentucky', '22': 'Louisiana', '23': 'Maine',
    '24': 'Maryland', '25': 'Massachusetts', '26': 'Michigan', '27': 'Minnesota', '28': 'Mississippi',
    '29': 'Missouri', '30': 'Montana', '31': 'Nebraska', '32': 'Nevada', '33': 'New Hampshire',
    '34': 'New Jersey', '35': 'New Mexico', '36': 'New York', '37': 'North Carolina', '38': 'North Dakota',
    '39': 'Ohio', '40': 'Oklahoma', '41': 'Oregon', '42': 'Pennsylvania', '44': 'Rhode Island',
    '45': 'South Carolina', '46': 'South Dakota', '47': 'Tennessee', '48': 'Texas', '49': 'Utah',
    '50': 'Vermont', '51': 'Virginia', '53': 'Washington', '54': 'West Virginia', '55': 'Wisconsin', 
    '56': 'Wyoming', '72': 'Puerto Rico', '78': 'Virgin Islands'
}

# Ensure outage_hourly has fips_code
if 'fips_code' not in outage_hourly.columns:
    if 'fips' in outage_hourly.columns:
        outage_hourly['fips_code'] = outage_hourly['fips'].astype(str).str.zfill(5)
    elif 'GEOID' in outage_hourly.columns:
        outage_hourly['fips_code'] = outage_hourly['GEOID'].astype(str).str.zfill(5)

# Build county-to-state mapping from counties_gdf
# counties_gdf has 'id' (county FIPS) and 'STATE' (two-digit state code)
counties_gdf['fips_code'] = counties_gdf['id'].astype(str).str.zfill(5)
counties_gdf['state_fips'] = counties_gdf['STATE'].astype(str).str.zfill(2)
counties_gdf['state_name'] = counties_gdf['state_fips'].map(fips_to_state)

# Create mapping from county FIPS to state name
county_to_state_name = counties_gdf.set_index('fips_code')['state_name'].to_dict()

# Map state names to outage data
outage_hourly['state_name'] = outage_hourly['fips_code'].map(county_to_state_name)

# Aggregate by datetime and state
state_hourly = (
    outage_hourly.dropna(subset=['state_name'])
    .groupby(['datetime','state_name'], as_index=False)[out_col]
    .sum()
)

print(f"States with outage data: {sorted(state_hourly['state_name'].unique())}")

# Plot: top 12 states by total outages
state_totals = state_hourly.groupby('state_name')[out_col].sum().sort_values(ascending=False)
keep_states = state_totals.head(12).index.tolist()
plot_df = state_hourly[state_hourly['state_name'].isin(keep_states)]

# Pivot for figure
pivot_df = plot_df.pivot(index='datetime', columns='state_name', values=out_col).fillna(0)
fig, ax = plt.subplots(figsize=(14, 6))
pivot_df.plot(ax=ax, lw=1.2, marker='o', markersize=2)
ax.set_title('Hourly outages by state (top 12 by total outages)', fontsize=14, fontweight='bold')
ax.set_xlabel('Time')
ax.set_ylabel('Customers out')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), title='State', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Below is where we do impact analyses (Ryan has much exploration, but not including in this cleaner code)